# quick look at the data

this is a quick tour of `cleaned_dataset.csv`. we'll check for blanks and repeats, then look at a few charts. it won't change the file or decide how the recommender works. open it from the project folder.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("cleaned_dataset.csv")

print("rows:", df.shape[0])
print("columns:", df.shape[1])
df.head()

## missing info and repeats

this shows any blank cells and repeated rows. it also checks if the same artist and song show up twice. repeats might be different versions, so don't delete them without checking.

In [ ]:
df.isnull().sum()

In [ ]:
print("exact duplicate rows:", df.duplicated().sum())
print("repeated artist and track pairs:", df[["Artist", "Track"]].duplicated().sum())

## a few sound details

here are a few song details we can use to find tracks that sound alike.

In [ ]:
df[["Danceability", "Energy", "Acousticness", "Valence", "Tempo", "Loudness"]].describe()

In [ ]:
# make the charts a bit bigger so the titles don't crowd each other
df[["Danceability", "Energy", "Acousticness", "Valence"]].hist(figsize=(9, 6), bins=20)
plt.suptitle("how the sound details are spread out")
plt.tight_layout()
plt.show()

## spotify streams

some songs have way more streams than others, so the bars get squished together. taking the log of the streams spreads them out. on the bottom axis, 6 means about a million streams, 7 means ten million, and 8 means a hundred million.

In [ ]:
# log10 turns huge numbers into small ones (+ 1 so a stream count of 0 doesn't break it)
log_streams = np.log10(df["Stream"].dropna() + 1)

plt.figure(figsize=(8, 5))
plt.hist(log_streams, bins=30, color="#1db954")
plt.xlabel("log10 of spotify streams (8 = 100 million)")
plt.ylabel("number of tracks")
plt.title("spotify streams")
plt.show()

## energy and streams

let's see if songs with more energy also tend to have more streams.

In [ ]:
plt.figure(figsize=(8, 5))

# small see-through dots so we can tell where lots of songs overlap
plt.scatter(df["Energy"], df["Stream"], s=5, alpha=0.3)

# log scale on streams so the popular songs don't squish everything else
plt.yscale("log")
plt.xlabel("energy")
plt.ylabel("spotify streams (log scale)")
plt.title("energy and spotify streams")
plt.show()

## spotify and youtube

do songs that get lots of streams on spotify also get lots of views on youtube? each dot is one song. if the dots slope upward, the answer is mostly yes.

In [ ]:
plt.figure(figsize=(7, 5))

# small see-through dots again, and both axes on a log scale
plt.scatter(df["Stream"], df["Views"], s=5, alpha=0.3)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("spotify streams (log scale)")
plt.ylabel("youtube views (log scale)")
plt.title("spotify streams and youtube views")
plt.show()

## do the most streamed songs sound different?

a boxplot shows where the middle half of the songs fall. the box is the middle half, the orange line is the middle song, and the dots are unusual songs far from the rest. we compare the top 10% of songs by streams to everything else.

In [ ]:
# the top 10% of songs by streams vs everyone else
cutoff = df["Stream"].quantile(0.9)
top = df[df["Stream"] >= cutoff]
rest = df[df["Stream"] < cutoff]

fig, axes = plt.subplots(1, 3, figsize=(10, 4), sharey=True)

for ax, feature in zip(axes, ["Energy", "Danceability", "Valence"]):
    ax.boxplot([rest[feature].dropna(), top[feature].dropna()])
    ax.set_xticklabels(["rest", "top 10%"])
    ax.set_title(feature)

axes[0].set_ylabel("value (0 to 1)")
plt.suptitle("do the most streamed songs sound different?")
plt.tight_layout()
plt.show()

## what the recommendations look like

here's one song and the three songs closest to it in sound. each group of bars is one sound detail, and each color is a song. if the colors are close together, the songs sound alike in that way.

In [ ]:
features = ["Danceability", "Energy", "Acousticness", "Valence", "Tempo", "Loudness"]

# put every sound detail on a 0 to 1 scale so tempo doesn't dwarf the rest
scaled = (df[features] - df[features].min()) / (df[features].max() - df[features].min())

# pick a song (the first match is the gorillaz one)
chosen = df[df["Track"] == "Feel Good Inc."].index[0]

# a smaller distance means the song sounds more like the one we picked
distances = ((scaled - scaled.loc[chosen]) ** 2).sum(axis=1) ** 0.5

# the closest rows, skipping the song itself (it's always first with a distance of 0)
rows = [chosen] + list(distances.sort_values().index[1:4])
compare = scaled.loc[rows]

# shorten long names so the legend stays readable
compare.index = [df.loc[row, "Track"][:20] for row in rows]

compare.T.plot(kind="bar", figsize=(10, 5))
plt.ylabel("value (scaled 0 to 1)")
plt.title("chosen song vs its 3 closest songs")
plt.xticks(rotation=0)

# leave some room at the top so the legend doesn't cover the bars
plt.ylim(0, 1.25)
plt.legend(ncol=2)
plt.show()

## quick reminder

these charts only cover the songs in this file. they can show which songs sound alike, but not whether someone would like a suggestion.